# 01 — Floating Point Basics

> Companion to **Week 11**, Part 1 of the lecture.

## What you will see

Computers can't store every decimal number perfectly. They use a clever trick
called **floating point** that approximates real numbers using a fixed number
of bits (32 bits = FP32, 16 bits = FP16, 8 bits for an integer = INT8).

By the end of this notebook you will be able to answer:

1. Why is `0.1 + 0.2` *not* exactly `0.3`?
2. What does the bit pattern of an FP32 number look like?
3. How much does an FP32 weight change when we round it to FP16 or "fake INT8"?


## Step 1 — The famous floating-point gotcha

Run the cell below. **Do not be alarmed** if Python tells you that `0.1 + 0.2`
is *not* equal to `0.3`. This is a feature of floating point, not a bug.


In [ ]:
import struct

result = 0.1 + 0.2
print("0.1 + 0.2 =", result)
print("Is it exactly 0.3?", result == 0.3)


**What happened?** Numbers like `0.1`, `0.2`, `0.3` cannot be written
exactly in binary — just like `1/3` cannot be written exactly in decimal.
The result is *very close* to `0.3`, but not exactly `0.3`.

> ⚠️ **Lesson for ML:** never compare floats with `==`. Use `abs(a - b) < 1e-6`
> or `numpy.isclose(a, b)` instead.

## Step 2 — Look at the actual bits

A 32-bit float is stored as a sign bit, an exponent, and a mantissa. We can
peek at the bit pattern using Python's `struct` module.


In [ ]:
for value in [0.0, 1.0, 0.1, 0.2, 0.3, -1.5]:
    bits = struct.unpack(">I", struct.pack(">f", value))[0]
    print(f"{value:>5}:  {bits:032b}")


Take a moment. Notice that:

- The first bit is the **sign**: 0 for positive, 1 for negative.
- The next 8 bits are the **exponent**.
- The last 23 bits are the **mantissa** (the significant digits).

You do **not** need to memorize this — just know that every float lives inside
32 of these little switches.

## Step 3 — FP32 vs FP16 vs (fake) INT8

Now let's pretend we are quantizing some neural-network weights. We will:

1. Start from a few FP32 values.
2. Round them to FP16.
3. Round them to a simple INT8 approximation (multiply by a scale, round,
   then divide back).
4. Compare how big each rounding error is.


In [ ]:
import numpy as np
import pandas as pd

weights = np.array([0.12, -1.78, 3.14159, 0.004, -0.56], dtype=np.float32)
fp16 = weights.astype(np.float16)

# Simple "fake INT8" using a scale factor
scale = 32
int8_values = np.clip(np.round(weights * scale), -128, 127).astype(np.int8)
recovered = int8_values.astype(np.float32) / scale

table = pd.DataFrame(
    {
        "fp32":      weights,
        "fp16":      fp16.astype(np.float32),
        "fake_int8": recovered,
    }
)
table["fp16_error"]      = (table["fp32"] - table["fp16"]).abs()
table["fake_int8_error"] = (table["fp32"] - table["fake_int8"]).abs()
table


### What to look for in that table

- The **fp16** column should look almost identical to **fp32** — errors near zero.
- The **fake_int8** column should be visibly chunkier — bigger steps between values.
- The **errors** are still very small for inference-style use.

That tiny error is the price you pay for using fewer bits per number. In return
you get a model that is 2× to 4× smaller in memory.

## 🧪 Your turn

Change the `scale` from `32` to `8` and re-run the cell. What happens to the
INT8 errors? Why do you think a smaller scale gives a chunkier approximation?
